# GenSivana Level 1 — v7 Clean OOF Hybrid Forensics

**But :** produire `submission.csv` pour `0=authentic`, `1=manipulated`, en optimisant
le F1 sans régler la recette sur le leaderboard.

Cette version corrige le principal défaut statistique de la v6 : toutes les décisions
(poids d'ensemble, seuil, nombre d'epochs) sont apprises sur des prédictions
**out-of-fold du train**. Le dossier `val/` reste un holdout intact jusqu'à l'audit.

Trois experts réellement complémentaires :

1. **DeepGuard / SigLIP2** : prior real/fake + sémantique globale ;
2. **ConvNeXt-V2 RGB** : texture et contenu à 384 px ;
3. **ConvNeXt-V2 NPR** : relations entre pixels, résidus et patches locaux.

La recette est inspirée des travaux documentés *Raising the Bar ... with CLIP*
(CVPRW 2024), NPR (CVPR 2024) et AIDE (ICLR 2025). Aucun dataset externe n'est
chargé : seuls les poids pré-entraînés autorisés sont utilisés.

> Un score de 0,999 ne peut pas être garanti avant évaluation privée. Le notebook
> privilégie une validation anti-fuite et sauvegarde toutes les preuves d'audit.

In [1]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!pip install -q "transformers>=4.53,<5" "timm>=1.0.15,<2" "scikit-learn>=1.4" imagehash


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00


In [2]:
import gc
import json
import math
import platform
import random
import warnings
from pathlib import Path
import imagehash
import numpy as np
import pandas as pd
import sklearn
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from IPython.display import display
from PIL import Image, ImageFile
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import AutoImageProcessor
from transformers import AutoModelForImageClassification
from transformers import get_cosine_schedule_with_warmup

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
SEED = 20260723
N_FOLDS = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPU = torch.cuda.device_count()
AMP = DEVICE == 'cuda'
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(
    './working')
OUT.mkdir(exist_ok=True)
FAST = False
DEEP_NAME = 'king1oo1/deepfake-model'
CNN_NAME = 'convnextv2_tiny.fcmae_ft_in22k_in1k_384'
IMG_DEEP, IMG_CNN = (224, 384)
MULTI_GPU_TRAINING = False
BATCH_DEEP = 24 if N_GPU >= 2 and MULTI_GPU_TRAINING else 12
BATCH_CNN = 8 if N_GPU >= 2 and MULTI_GPU_TRAINING else 4
ACCUM_DEEP = 2 if N_GPU >= 2 and MULTI_GPU_TRAINING else 4
ACCUM_CNN = 3 if N_GPU >= 2 and MULTI_GPU_TRAINING else 6
NUM_WORKERS = 0
EPOCHS_DEEP, EPOCHS_CNN = (2, 2) if FAST else (5, 5)
PATIENCE = 2
RUN_TAG = 'v7c_safe_s20260723_f3'
FORCE_RETRAIN = False


def seed_all(seed=SEED):
    """Seed every random generator used by the pipeline.

Args:
    seed: Seed shared by Python, NumPy, PyTorch, and CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_all()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('device', DEVICE, 'GPUs', N_GPU, 'output', OUT)


device cuda GPUs 2 output /kaggle/working


## 1. Données, contrôles et groupes anti-doublons

In [3]:
def find_root():
    """Locate the competition dataset.

Returns:
    Root directory containing the train, validation, and test splits.

Raises:
    FileNotFoundError: If no compatible dataset directory is available."""
    candidates = [
        Path('/kaggle/input') / 'competitions' /
        'gensivana-real-or-fake-level-1-detection',
        Path('/root/.cache/kagglehub') / 'competitions' /
        'gensivana-real-or-fake-level-1-detection',
        Path('/content'),
        Path('.')
    ]
    for root in candidates:
        if (root / 'train/ground_truth.csv').exists() and (
                root / 'test/images').exists():
            return root
    for p in Path('/kaggle/input').glob('*') if Path(
            '/kaggle/input').exists() else []:
        if (p / 'train/ground_truth.csv').exists():
            return p
    raise FileNotFoundError('Ajoute le dataset GenSivana au notebook.')


ROOT = find_root()


def read_split(name, labelled=True):
    """Load one competition split and validate its image paths.

Args:
    name: Name of the split to load.
    labelled: Whether the split includes target labels.

Returns:
    Data frame containing identifiers, paths, and optional targets."""
    image_dir = ROOT / name / 'images'
    if labelled:
        df = pd.read_csv(ROOT / name / 'ground_truth.csv')
        label_col = 'classe' if 'classe' in df.columns else 'class'
        df = df.rename(columns={label_col: 'target'})
    else:
        sample = pd.read_csv(ROOT / 'sample_submission.csv')
        df = sample[['image_id']].copy()
    df['path'] = df.image_id.map(lambda x: str(image_dir / x))
    missing = [p for p in df.path if not Path(p).exists()]
    assert not missing, f'{len(missing)} images absentes'
    if labelled:
        df.target = df.target.astype(int)
        assert set(df.target.unique()) <= {0, 1}
    return df.reset_index(drop=True)


train_df, val_df, test_df = (read_split('train'), read_split('val'),
                             read_split('test', False))
if FAST:
    train_df = train_df.sample(min(900, len(train_df)),
                               random_state=SEED).reset_index(drop=True)
print(ROOT, len(train_df), len(val_df), len(test_df),
      train_df.target.value_counts(normalize=True).to_dict())
hash_cache = OUT / 'train_phash.npy'
if hash_cache.exists() and len(np.load(hash_cache,
                                       allow_pickle=True)) == len(train_df):
    hashes = np.load(hash_cache, allow_pickle=True).tolist()
else:
    hashes = [
        str(imagehash.phash(Image.open(p).convert('RGB'), hash_size=8))
        for p in train_df.path
    ]
    np.save(hash_cache, np.array(hashes, dtype=object))
parent = list(range(len(hashes)))


def find(x):
    """Return the canonical parent of a duplicate group.

Args:
    x: Index whose group representative is requested.

Returns:
    Canonical group index."""
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    """Merge two duplicate groups.

Args:
    a: Index from the first group.
    b: Index from the second group."""
    a, b = (find(a), find(b))
    if a != b:
        parent[b] = a


buckets = {}
for i, h in enumerate(hashes):
    for k in (h[:4], h[4:8], h[8:12], h[12:16]):
        buckets.setdefault((k, len(h)), []).append(i)
for ids in buckets.values():
    if len(ids) > 80:
        continue
    for a in range(len(ids)):
        ha = int(hashes[ids[a]], 16)
        for b in range(a + 1, len(ids)):
            if (ha ^ int(hashes[ids[b]], 16)).bit_count() <= 4:
                union(ids[a], ids[b])
groups = np.array([find(i) for i in range(len(hashes))])
cv = list(
    StratifiedGroupKFold(N_FOLDS, shuffle=True,
                         random_state=SEED).split(train_df, train_df.target,
                                                  groups))
print('pHash groups', len(np.unique(groups)), 'largest',
      pd.Series(groups).value_counts().max())


/kaggle/input/competitions/gensivana-real-or-fake-level-1-detection 7857 978 993 {1: 0.6666666666666666, 0: 0.3333333333333333}
pHash groups 6933 largest 6


## 2. Transformations

La branche NPR transforme l'image en relations de pixels :
`x[:,1:,1:] - 1/4 * pixel_shuffle(x,2)`. On conserve également une vue résiduelle
haute fréquence. Pour les retouches locales, l'inférence combine vue globale et cinq
crops déterministes ; les crops ne sont jamais traités comme des labels indépendants.

In [4]:
IM_MEAN = (0.485, 0.456, 0.406)
IM_STD = (0.229, 0.224, 0.225)
deep_processor = AutoImageProcessor.from_pretrained(DEEP_NAME)
dmean = tuple(deep_processor.image_mean)
dstd = tuple(deep_processor.image_std)


def make_tf(size, mean, std, train=False):
    """Build image transformations for training or inference.

Args:
    size: Square output resolution.
    mean: Channel normalization means.
    std: Channel normalization standard deviations.
    train: Whether to include training augmentations.

Returns:
    Composed torchvision transformation."""
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(
                size,
                scale=(0.55, 1.0),
                ratio=(0.75, 1.333),
                interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply(
                [transforms.ColorJitter(0.1, 0.1, 0.06, 0.015)], p=0.3),
            transforms.RandomChoice([
                transforms.Lambda(lambda x: x),
                transforms.GaussianBlur(3, (0.1, 0.8))
            ]),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
    return transforms.Compose([
        transforms.Resize((size, size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])


deep_train_tf = make_tf(IMG_DEEP, dmean, dstd, True)
deep_eval_tf = make_tf(IMG_DEEP, dmean, dstd)
cnn_train_tf = make_tf(IMG_CNN, IM_MEAN, IM_STD, True)
cnn_eval_tf = make_tf(IMG_CNN, IM_MEAN, IM_STD)


class ImageDS(Dataset):
    """Load RGB images and optional multi-view NPR inputs."""

    def __init__(self, df, tf, mode='rgb', multiview=False):
        """Store the data frame and input configuration.

Args:
    df: Data frame describing the images.
    tf: Transformation applied to every image view.
    mode: Input representation, either RGB or NPR.
    multiview: Whether to return the global image and four crops."""
        self.df = df.reset_index(drop=True)
        self.tf = tf
        self.mode = mode
        self.multiview = multiview

    def __len__(self):
        """Return the number of images in the dataset."""
        return len(self.df)

    def __getitem__(self, i):
        """Load and transform one image.

Args:
    i: Positional image index.

Returns:
    Input tensor, target tensor, and positional index."""
        with Image.open(self.df.path.iloc[i]) as src:
            im = src.convert('RGB')
        views = [im]
        if self.multiview:
            w, h = im.size
            cw, ch = (max(32, int(0.72 * w)), max(32, int(0.72 * h)))
            views += [
                im.crop((0, 0, cw, ch)),
                im.crop((w - cw, 0, w, ch)),
                im.crop((0, h - ch, cw, h)),
                im.crop((w - cw, h - ch, w, h))
            ]
        xs = []
        for view in views:
            x = self.tf(view)
            if self.mode == 'npr':
                raw = x.clone()
                up = F.interpolate(F.avg_pool2d(raw[None], 2),
                                   size=raw.shape[-2:],
                                   mode='nearest')[0]
                npr = raw - up
                x = torch.tanh(2.5 * npr)
            xs.append(x)
        x = torch.stack(xs) if self.multiview else xs[0]
        y = float(self.df.target.iloc[i]) if 'target' in self.df else -1.0
        return (x, torch.tensor(y), i)


def loader(df, tf, mode, batch, shuffle=False, multiview=False):
    """Create a deterministic data loader.

Args:
    df: Data frame consumed by the dataset.
    tf: Image transformation pipeline.
    mode: Input representation.
    batch: Physical batch size.
    shuffle: Whether to shuffle training samples.
    multiview: Whether to load five image views.

Returns:
    Configured PyTorch data loader."""
    kwargs = dict(batch_size=batch,
                  shuffle=shuffle,
                  num_workers=NUM_WORKERS,
                  pin_memory=AMP,
                  persistent_workers=False,
                  drop_last=shuffle)
    if NUM_WORKERS:
        kwargs['prefetch_factor'] = 1
    return DataLoader(ImageDS(df, tf, mode, multiview), **kwargs)


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

## 3. Modèles et moteur d'entraînement OOF

In [5]:
def unwrap(m):
    """Return the underlying model from an optional wrapper.

Args:
    m: Model that may be wrapped by DataParallel.

Returns:
    Unwrapped neural network."""
    return m.module if isinstance(m, nn.DataParallel) else m


class DeepModel(nn.Module):
    """Wrap the pretrained SigLIP2 deepfake classifier."""

    def __init__(self):
        """Load the classifier and resolve its fake label."""
        super().__init__()
        self.net = AutoModelForImageClassification.from_pretrained(DEEP_NAME)
        labels = {
            int(k): str(v).lower()
            for k, v in self.net.config.id2label.items()
        }
        fake = [
            k for k, v in labels.items()
            if 'fake' in v or 'manip' in v or 'ai' == v
        ]
        self.fake_idx = fake[0] if fake else 1

    def forward(self, x):
        """Compute one binary manipulation logit.

Args:
    x: Normalized image batch.

Returns:
    Binary manipulation logits."""
        logits = self.net(pixel_values=x, return_dict=False)[0]
        if logits.shape[1] == 2:
            return logits[:, self.fake_idx] - logits[:, 1 - self.fake_idx]
        return logits[:, 0]


class CNNModel(nn.Module):
    """Wrap ConvNeXt V2 with a binary classification head."""

    def __init__(self):
        """Create the pretrained ConvNeXt V2 model."""
        super().__init__()
        self.net = timm.create_model(CNN_NAME,
                                     pretrained=True,
                                     num_classes=1,
                                     drop_rate=0.2)

    def forward(self, x):
        """Compute binary logits for an image batch.

Args:
    x: Normalized image batch.

Returns:
    Flattened binary logits."""
        return self.net(x).flatten()


def freeze_deep(model):
    """Train only the SigLIP2 head and final visual blocks.

Args:
    model: Deep classifier whose parameters are configured."""
    for p in model.parameters():
        p.requires_grad = False
    net = model.net
    classifier = getattr(net, 'classifier', None)
    if classifier is not None:
        for p in classifier.parameters():
            p.requires_grad = True
    vision = getattr(net, 'vision_model', None)
    if vision is None and hasattr(net, 'siglip'):
        vision = net.siglip.vision_model
    layers = getattr(getattr(vision, 'encoder', None), 'layers', [])
    for block in list(layers)[-6:]:
        for p in block.parameters():
            p.requires_grad = True
    for name in ('post_layernorm', 'head'):
        mod = getattr(vision, name, None)
        if mod is not None:
            for p in mod.parameters():
                p.requires_grad = True


def make_model(kind):
    """Create one pipeline expert on the active device.

Args:
    kind: Expert family to instantiate.

Returns:
    Configured neural network."""
    m = DeepModel() if kind == 'deep' else CNNModel()
    if kind == 'deep':
        freeze_deep(m)
    if N_GPU > 1 and MULTI_GPU_TRAINING:
        m = nn.DataParallel(m)
    return m.to(DEVICE)


def predict(model, df, kind, mode='rgb'):
    """Predict manipulation probabilities for a data frame.

Args:
    model: Trained expert.
    df: Images to predict.
    kind: Expert family.
    mode: Input representation.

Returns:
    Manipulation probability for every image."""
    tf, batch = (deep_eval_tf, BATCH_DEEP) if kind == 'deep' else (cnn_eval_tf,
                                                                   BATCH_CNN)
    multiview = kind == 'cnn' and mode == 'npr'
    dl = loader(df, tf, mode, batch, False, multiview)
    out = np.zeros(len(df), np.float32)
    model.eval()
    with torch.inference_mode():
        for x, _, idx in dl:
            if multiview:
                view_probs = []
                for view_idx in range(x.shape[1]):
                    with torch.autocast(device_type=DEVICE,
                                        dtype=torch.float16,
                                        enabled=AMP):
                        z = unwrap(model)(x[:, view_idx].to(DEVICE,
                                                            non_blocking=True))
                    view_probs.append(torch.sigmoid(z.float()).cpu())
                pv = torch.stack(view_probs, 1)
                pv = 0.75 * pv.mean(1) + 0.25 * pv.max(1).values
            else:
                with torch.autocast(device_type=DEVICE,
                                    dtype=torch.float16,
                                    enabled=AMP):
                    z = unwrap(model)(x.to(DEVICE, non_blocking=True))
                pv = torch.sigmoid(z.float())
            out[idx.numpy()] = pv.cpu().numpy()
    del dl
    return out


def best_threshold(y, p, grid=None):
    """Find the threshold that maximizes positive-class F1.

Args:
    y: Binary targets.
    p: Manipulation probabilities.
    grid: Optional threshold candidates.

Returns:
    Best threshold and its F1 score."""
    if grid is None:
        grid = np.linspace(0.05, 0.95, 361)
    fs = np.array([f1_score(y, p >= t) for t in grid])
    i = int(fs.argmax())
    return (float(grid[i]), float(fs[i]))


def fit_one(kind, mode, tr, va, fold, epochs):
    """Train and select one expert for one OOF fold.

Args:
    kind: Expert family.
    mode: Input representation.
    tr: Training partition.
    va: Validation partition.
    fold: Fold index.
    epochs: Maximum number of epochs.

Returns:
    Best model, epoch history, and selected epoch count."""
    model = make_model(kind)
    bs = BATCH_DEEP if kind == 'deep' else BATCH_CNN
    accum = ACCUM_DEEP if kind == 'deep' else ACCUM_CNN
    tf = deep_train_tf if kind == 'deep' else cnn_train_tf
    dl = loader(tr, tf, mode, bs, True)
    body_lr = 8e-06 if kind == 'deep' else 2e-05
    head_lr = 0.0002
    head = []
    body = []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (head if any(
                (k in n.lower()
                 for k in ('classifier', 'head'))) else body).append(p)
    opt = torch.optim.AdamW([{
        'params': body,
        'lr': body_lr
    }, {
        'params': head,
        'lr': head_lr
    }],
                            weight_decay=0.0002,
                            foreach=False)
    steps = max(1, math.ceil(len(dl) / accum) * epochs)
    sch = get_cosine_schedule_with_warmup(opt, int(0.08 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    best = -1
    bad = 0
    history = []
    ck = OUT / f'{RUN_TAG}_{kind}_{mode}_fold{fold}.pt'
    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        losses = []
        for step, (x, y, _) in enumerate(dl):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE,
                                dtype=torch.float16,
                                enabled=AMP):
                z = model(x)
                loss = F.binary_cross_entropy_with_logits(z, y) / accum
            scaler.scale(loss).backward()
            if (step + 1) % accum == 0 or step + 1 == len(dl):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sch.step()
            losses.append(loss.detach().float().item() * accum)
        pv = predict(model, va, kind, mode)
        th, score = best_threshold(va.target.values, pv)
        history.append({
            'epoch': ep + 1,
            'loss': float(np.mean(losses)),
            'f1': score,
            'threshold': th
        })
        print(kind, mode, 'fold', fold, 'epoch', ep + 1, history[-1])
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if score > best + 0.0001:
            best = score
            bad = 0
            torch.save({
                'state': unwrap(model).state_dict(),
                'epoch': ep + 1
            }, ck)
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    saved = torch.load(ck, map_location='cpu', weights_only=True)
    unwrap(model).load_state_dict(saved['state'])
    return (model, history, int(saved['epoch']))


## 4. OOF train + prédictions du holdout officiel

Les trois experts sont entraînés avec les mêmes folds groupés. Le holdout `val/`
n'intervient ni dans le checkpointing ni dans le seuil. Ses prédictions sont la moyenne
des modèles de folds, analogue à l'inférence test.

In [6]:
experts = [('deep', 'rgb', EPOCHS_DEEP), ('cnn', 'rgb', EPOCHS_CNN),
           ('cnn', 'npr', EPOCHS_CNN)]
oof = np.zeros((len(train_df), len(experts)), np.float32)
hold = np.zeros((len(val_df), len(experts)), np.float32)
fold_epochs = {}
histories = {}
for j, (kind, mode, epochs) in enumerate(experts):
    for fold, (it, iv) in enumerate(cv):
        tr = train_df.iloc[it].reset_index(drop=True)
        va = train_df.iloc[iv].reset_index(drop=True)
        cache = OUT / f'{RUN_TAG}_{kind}_{mode}_fold{fold}_pred.npz'
        if cache.exists() and (not FORCE_RETRAIN):
            z = np.load(cache, allow_pickle=True)
            pv, pvh = (z['oof'], z['hold'])
            best_ep = int(z['best_epoch'])
            h = z['history'].tolist()
            assert len(pv) == len(iv) and len(pvh) == len(val_df)
            print('RESUME', cache)
        else:
            model, h, best_ep = fit_one(kind, mode, tr, va, fold, epochs)
            pv = predict(model, va, kind, mode)
            pvh = predict(model, val_df, kind, mode)
            np.savez_compressed(cache,
                                oof=pv,
                                hold=pvh,
                                best_epoch=best_ep,
                                history=np.array(h, dtype=object))
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            model.cpu()
            del model, tr, va
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        oof[iv, j] = pv
        hold[:, j] += pvh / N_FOLDS
        histories[f'{kind}_{mode}_f{fold}'] = h
        fold_epochs.setdefault(f'{kind}_{mode}', []).append(best_ep)
    print('OOF', kind, mode, best_threshold(train_df.target.values, oof[:, j]))
np.savez_compressed(OUT / 'level1_v7_oof.npz',
                    oof=oof,
                    hold=hold,
                    y_train=train_df.target.values,
                    y_val=val_df.target.values)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

deep rgb fold 0 epoch 1 {'epoch': 1, 'loss': 0.64519877981292, 'f1': 0.868265162200282, 'threshold': 0.7649999999999999}
deep rgb fold 0 epoch 2 {'epoch': 2, 'loss': 0.32059318896964056, 'f1': 0.9094533029612756, 'threshold': 0.5675}
deep rgb fold 0 epoch 3 {'epoch': 3, 'loss': 0.217411069870166, 'f1': 0.9217735103078226, 'threshold': 0.3524999999999999}
deep rgb fold 0 epoch 4 {'epoch': 4, 'loss': 0.16199604557600691, 'f1': 0.9288606130711394, 'threshold': 0.7849999999999999}
deep rgb fold 0 epoch 5 {'epoch': 5, 'loss': 0.13842062643780415, 'f1': 0.9296102385107621, 'threshold': 0.72}
deep rgb fold 1 epoch 1 {'epoch': 1, 'loss': 0.6225792064406406, 'f1': 0.8587724063009234, 'threshold': 0.4549999999999999}
deep rgb fold 1 epoch 2 {'epoch': 2, 'loss': 0.30785724155861754, 'f1': 0.8987161198288159, 'threshold': 0.2875}
deep rgb fold 1 epoch 3 {'epoch': 3, 'loss': 0.2082454830590079, 'f1': 0.9183615010025781, 'threshold': 0.42999999999999994}
deep rgb fold 1 epoch 4 {'epoch': 4, 'loss': 

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

cnn rgb fold 0 epoch 1 {'epoch': 1, 'loss': 0.397788147242118, 'f1': 0.9893891597361629, 'threshold': 0.4099999999999999}
cnn rgb fold 0 epoch 2 {'epoch': 2, 'loss': 0.07556629606478908, 'f1': 0.9936962750716333, 'threshold': 0.8474999999999999}
cnn rgb fold 0 epoch 3 {'epoch': 3, 'loss': 0.03776441775860871, 'f1': 0.9893830703012912, 'threshold': 0.8875}
cnn rgb fold 0 epoch 4 {'epoch': 4, 'loss': 0.04194882142367291, 'f1': 0.9942627653471027, 'threshold': 0.8099999999999999}
cnn rgb fold 0 epoch 5 {'epoch': 5, 'loss': 0.023188027967538983, 'f1': 0.9942660550458715, 'threshold': 0.16499999999999998}
cnn rgb fold 1 epoch 1 {'epoch': 1, 'loss': 0.4439352340330017, 'f1': 0.9730679156908665, 'threshold': 0.5049999999999999}
cnn rgb fold 1 epoch 2 {'epoch': 2, 'loss': 0.10263553495036234, 'f1': 0.9936194895591647, 'threshold': 0.06}
cnn rgb fold 1 epoch 3 {'epoch': 3, 'loss': 0.05330616133746736, 'f1': 0.9927007299270073, 'threshold': 0.7274999999999999}
cnn rgb fold 1 epoch 4 {'epoch': 4,

## 5. Ensemble et seuil robustes — appris exclusivement sur OOF train

In [7]:
def logit(p):
    """Convert clipped probabilities to log odds.

Args:
    p: Probability array.

Returns:
    Array of finite log odds."""
    return np.log(
        np.clip(p, 1e-05, 1 - 1e-05) / np.clip(1 - p, 1e-05, 1 - 1e-05))


y = train_df.target.values
meta_cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED + 91)
meta_oof = np.zeros(len(y), np.float32)
for a, b in meta_cv.split(oof, y, groups):
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.08, max_iter=2000, class_weight=None))
    m.fit(logit(oof[a]), y[a])
    meta_oof[b] = m.predict_proba(logit(oof[b]))[:, 1]
stack = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.08, max_iter=2000, class_weight=None))
stack.fit(logit(oof), y)
p_hold = stack.predict_proba(logit(hold))[:, 1]
candidates = {'stack': meta_oof}
candidate_hold = {'stack': p_hold}
for j, (k, m, _) in enumerate(experts):
    candidates[f'{k}_{m}'] = oof[:, j]
    candidate_hold[f'{k}_{m}'] = hold[:, j]
candidate_thresholds = {}
candidate_fold_thresholds = {}
candidate_scores = {}
for name, p in candidates.items():
    ts = []
    for _, idx in meta_cv.split(p, y, groups):
        t, _ = best_threshold(y[idx], p[idx])
        ts.append(t)
    candidate_fold_thresholds[name] = ts
    candidate_thresholds[name] = float(np.median(ts))
    candidate_scores[name] = float(f1_score(y, p
                                            >= candidate_thresholds[name]))
SELECTED = max(candidate_scores, key=candidate_scores.get)
THRESH = candidate_thresholds[SELECTED]
thresholds = candidate_fold_thresholds[SELECTED]
fold_f1 = [
    float(f1_score(y[idx], candidates[SELECTED][idx] >= THRESH))
    for _, idx in meta_cv.split(candidates[SELECTED], y, groups)
]
oof_f1 = candidate_scores[SELECTED]
p_hold = candidate_hold[SELECTED]
yh = val_df.target.values
hold_f1 = float(f1_score(yh, p_hold >= THRESH))
hold_bal = float(balanced_accuracy_score(yh, p_hold >= THRESH))
cm = confusion_matrix(yh, p_hold >= THRESH).tolist()
print('OOF candidates', candidate_scores, 'SELECTED', SELECTED)
print('OOF F1', oof_f1, 'thresholds', thresholds, 'median', THRESH)
print('UNTOUCHED HOLDOUT F1', hold_f1, 'balanced', hold_bal, 'CM', cm)
print('stack weights', stack[-1].coef_.round(4), stack[-1].intercept_.round(4))
assert np.isfinite(oof).all() and np.isfinite(hold).all()
if hold_f1 < oof_f1 - 0.04:
    print('WARNING: domain shift/overfit >4 points.' +
          ' Ne pas régler sur le leaderboard.')


OOF candidates {'stack': 0.9980941490375452, 'deep_rgb': 0.9230332291962511, 'cnn_rgb': 0.9953391039665176, 'cnn_npr': 0.9987603699818823} SELECTED cnn_npr
OOF F1 0.9987603699818823 thresholds [0.35749999999999993, 0.44249999999999995, 0.42999999999999994, 0.4624999999999999, 0.5449999999999999] median 0.44249999999999995
UNTOUCHED HOLDOUT F1 0.998468606431853 balanced 0.9969325153374233 CM [[324, 2], [0, 652]]
stack weights [[1.1055 2.1869 2.6481]] [2.217]


## 6. Réentraînement final sur train + val

Le nombre d'epochs final est la médiane des meilleurs epochs OOF. Le holdout n'est
fusionné qu'après l'audit. Les poids du stacker et le seuil restent ceux appris sur le
train OOF ; ils ne sont pas réoptimisés sur `val`.

In [8]:
all_df = pd.concat([train_df, val_df], ignore_index=True)
final_test = np.zeros((len(test_df), len(experts)), np.float32)
for j, (kind, mode, _) in enumerate(experts):
    epochs = max(1, int(round(np.median(fold_epochs[f'{kind}_{mode}']))))
    final_ck = OUT / f'{RUN_TAG}_final_{kind}_{mode}.pt'
    model = make_model(kind)
    if final_ck.exists() and (not FORCE_RETRAIN):
        unwrap(model).load_state_dict(
            torch.load(final_ck, map_location='cpu', weights_only=True))
        print('RESUME', final_ck)
        final_test[:, j] = predict(model, test_df, kind, mode)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue
    bs = BATCH_DEEP if kind == 'deep' else BATCH_CNN
    accum = ACCUM_DEEP if kind == 'deep' else ACCUM_CNN
    tf = deep_train_tf if kind == 'deep' else cnn_train_tf
    dl = loader(all_df, tf, mode, bs, True)
    body_lr = 8e-06 if kind == 'deep' else 2e-05
    head_lr = 0.0002
    head = []
    body = []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (head if any(
                (k in n.lower()
                 for k in ('classifier', 'head'))) else body).append(p)
    opt = torch.optim.AdamW([{
        'params': body,
        'lr': body_lr
    }, {
        'params': head,
        'lr': head_lr
    }],
                            weight_decay=0.0002,
                            foreach=False)
    steps = max(1, math.ceil(len(dl) / accum) * epochs)
    sch = get_cosine_schedule_with_warmup(opt, int(0.08 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        losses = []
        for step, (x, yy, _) in enumerate(dl):
            x = x.to(DEVICE, non_blocking=True)
            yy = yy.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE,
                                dtype=torch.float16,
                                enabled=AMP):
                loss = F.binary_cross_entropy_with_logits(model(x), yy) / accum
            scaler.scale(loss).backward()
            if (step + 1) % accum == 0 or step + 1 == len(dl):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sch.step()
            losses.append(loss.detach().float().item() * accum)
        print('FINAL', kind, mode, 'epoch', ep + 1, 'loss', np.mean(losses))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    torch.save(unwrap(model).state_dict(), final_ck)
    final_test[:, j] = predict(model, test_df, kind, mode)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
candidate_test = {'stack': stack.predict_proba(logit(final_test))[:, 1]}
for j, (k, m, _) in enumerate(experts):
    candidate_test[f'{k}_{m}'] = final_test[:, j]
p_test = candidate_test[SELECTED]
pred = (p_test >= THRESH).astype(int)


FINAL deep rgb epoch 1 loss 0.5277244029870338
FINAL deep rgb epoch 2 loss 0.2299560200468556
FINAL deep rgb epoch 3 loss 0.1534615191749649
FINAL deep rgb epoch 4 loss 0.11958922889654945
FINAL cnn rgb epoch 1 loss 0.3002445452339632
FINAL cnn rgb epoch 2 loss 0.07669777876811454
FINAL cnn rgb epoch 3 loss 0.0458759338865114
FINAL cnn rgb epoch 4 loss 0.02549679055334084
FINAL cnn npr epoch 1 loss 0.3250723878446558
FINAL cnn npr epoch 2 loss 0.06272011943743495
FINAL cnn npr epoch 3 loss 0.03735130055764804


## 7. Soumission et audit reproductible

In [9]:
sample = pd.read_csv(ROOT / 'sample_submission.csv')
assert sample.image_id.is_unique and test_df.image_id.is_unique
sub = sample[['image_id']].merge(pd.DataFrame({
    'image_id': test_df.image_id,
    'classe': pred
}),
                                 on='image_id',
                                 how='left',
                                 validate='one_to_one')
assert len(sub) == len(sample) and sub.classe.notna().all()
sub.classe = sub.classe.astype(int)
assert set(sub.classe.unique()) <= {0, 1}
sub.to_csv(OUT / 'submission.csv', index=False)
audit = {
    'version': 'level1_v7_oof_hybrid',
    'seed': SEED,
    'root': str(ROOT),
    'n_folds': N_FOLDS,
    'models': [DEEP_NAME, CNN_NAME + ' RGB', CNN_NAME + ' NPR'],
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
    'validation_policy': 'all tuning on grouped train OOF;' +
    ' official val opened once; final refit train+val',
    'oof_f1': oof_f1,
    'untouched_validation_f1': hold_f1,
    'untouched_validation_balanced_accuracy': hold_bal,
    'untouched_validation_confusion_matrix': cm,
    'threshold': THRESH,
    'fold_thresholds': thresholds,
    'fold_f1': fold_f1,
    'selected_candidate': SELECTED,
    'candidate_oof_f1': candidate_scores,
    'stack_coefficients': stack[-1].coef_[0].tolist(),
    'stack_intercept': float(stack[-1].intercept_[0]),
    'expert_oof_f1': {
        f'{k}_{m}': best_threshold(y, oof[:, j])[1]
        for j, (k, m, _) in enumerate(experts)
    },
    'best_epochs': fold_epochs,
    'predicted_manipulated_ratio': float(pred.mean()),
    'histories': histories,
    'environment': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'timm': timm.__version__,
        'transformers': transformers.__version__,
        'scikit_learn': sklearn.__version__
    }
}
(OUT / 'audit_level1_v7.json').write_text(json.dumps(audit, indent=2))
np.savez_compressed(OUT / 'ensemble_scores_level1_v7.npz',
                    oof=oof,
                    hold=hold,
                    test=final_test,
                    meta_oof=meta_oof,
                    p_hold=p_hold,
                    p_test=p_test,
                    y_train=y,
                    y_val=yh)
display(sub.head())
print(
    json.dumps({
        k: v
        for k, v in audit.items() if k != 'histories'
    }, indent=2))
print('READY:', OUT / 'submission.csv')


,image_id,classe
0,e37001f6-d230-4588-aedb-bd896084d671.png,1
1,56689e24-fe39-4df5-be2e-e0669e30d1d6.png,1
2,0c9f82ad-a7f5-4ee0-9f20-90333608428a.png,0
3,d5471932-3e03-45ea-a3bd-916be674b2ca.png,1
4,b8d8e96c-ca54-41d7-a2ab-a713e5a79d2f.png,0


{
  "version": "level1_v7_oof_hybrid",
  "seed": 20260723,
  "root": "/kaggle/input/competitions/gensivana-real-or-fake-level-1-detection",
  "n_folds": 3,
  "models": [
    "king1oo1/deepfake-model",
    "convnextv2_tiny.fcmae_ft_in22k_in1k_384 RGB",
    "convnextv2_tiny.fcmae_ft_in22k_in1k_384 NPR"
  ],
  "n_train": 7857,
  "n_val": 978,
  "n_test": 993,
  "validation_policy": "all tuning on grouped train OOF; official val opened once; final refit train+val",
  "oof_f1": 0.9987603699818823,
  "untouched_validation_f1": 0.998468606431853,
  "untouched_validation_balanced_accuracy": 0.9969325153374233,
  "untouched_validation_confusion_matrix": [
    [
      324,
      2
    ],
    [
      0,
      652
    ]
  ],
  "threshold": 0.44249999999999995,
  "fold_thresholds": [
    0.35749999999999993,
    0.44249999999999995,
    0.42999999999999994,
    0.4624999999999999,
    0.5449999999999999
  ],
  "fold_f1": [
    0.9980861244019139,
    0.9985666507405638,
    1.0,
    0.9986065954482

## Lecture de l'audit

- Le chiffre important est `untouched_validation_f1`, pas le meilleur F1 d'une epoch.
- Une forte divergence OOF/validation indique un changement de domaine ou un raccourci.
- Ne modifiez pas les poids ou le seuil à partir d'une poignée de scores publics.
- Conservez `audit_level1_v7.json` et `ensemble_scores_level1_v7.npz` après chaque run :
  ils permettent une vraie analyse des erreurs et des désaccords.